In [0]:
# Importações e caminho base do Volume (Unity Catalog) com os CSVs brutos do Olist.
from pyspark.sql.functions import current_timestamp, lit

volume_path = "/Volumes/ecommerce_br/bronze/raw_files"

In [0]:
# Mapeamento entre cada CSV bruto e o nome da respectiva tabela Bronze.
# Nomes de tabela enxutos: o catálogo e o schema já fornecem o contexto (ecommerce_br.bronze).
arquivos_para_ingestao = {
    "olist_customers_dataset.csv": "customers",
    "olist_geolocation_dataset.csv": "geolocation",
    "olist_orders_dataset.csv": "orders",
    "olist_order_items_dataset.csv": "order_items",
    "olist_order_payments_dataset.csv": "order_payments",
    "olist_order_reviews_dataset.csv": "order_reviews",
    "olist_products_dataset.csv": "products",
    "olist_sellers_dataset.csv": "sellers",
    "product_category_name_translation.csv": "category_translation",
}

In [0]:
# Leitura do CSV bruto. inferSchema=False lê todas as colunas como texto:
# o Bronze preserva o dado exatamente como veio do CSV (a tipagem correta
# é responsabilidade da camada Silver). Isso também evita inferências de
# tipo inconsistentes entre execuções.
# multiLine + quote/escape garantem o parsing correto de campos de texto
# que contêm quebras de linha e vírgulas (ex.: comentários em order_reviews),
# evitando que um registro seja quebrado em várias linhas.
for nome_arquivo, nome_tabela in arquivos_para_ingestao.items():

    # Leitura do CSV bruto. inferSchema deduz os tipos na ingestão inicial.
    df = spark.read.csv(
        f"{volume_path}/{nome_arquivo}",
        header=True,
        inferSchema=False,
        multiLine=True,
        quote='"',
        escape='"'
    )

    # Metadados de controle da camada Bronze (rastreabilidade da carga).
    # Os dados originais são preservados; apenas duas colunas são acrescentadas.
    df_bronze = df \
        .withColumn("data_ingestao", current_timestamp()) \
        .withColumn("arquivo_origem", lit(nome_arquivo))

    # Persistência como tabela Delta. mode overwrite torna a ingestão idempotente.
    # overwriteSchema=True substitui também o schema quando a tabela já existe
    # (necessário aqui porque a ingestão passou a ler todas as colunas como texto).
    df_bronze.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"ecommerce_br.bronze.{nome_tabela}")

    print(f"OK: {nome_arquivo} -> ecommerce_br.bronze.{nome_tabela} ({df_bronze.count()} linhas)")

print("\nIngestão Bronze concluída para todos os arquivos.")

OK: olist_customers_dataset.csv -> ecommerce_br.bronze.customers (99441 linhas)
OK: olist_geolocation_dataset.csv -> ecommerce_br.bronze.geolocation (1000163 linhas)
OK: olist_orders_dataset.csv -> ecommerce_br.bronze.orders (99441 linhas)
OK: olist_order_items_dataset.csv -> ecommerce_br.bronze.order_items (112650 linhas)
OK: olist_order_payments_dataset.csv -> ecommerce_br.bronze.order_payments (103886 linhas)
OK: olist_order_reviews_dataset.csv -> ecommerce_br.bronze.order_reviews (99224 linhas)
OK: olist_products_dataset.csv -> ecommerce_br.bronze.products (32951 linhas)
OK: olist_sellers_dataset.csv -> ecommerce_br.bronze.sellers (3095 linhas)
OK: product_category_name_translation.csv -> ecommerce_br.bronze.category_translation (71 linhas)

Ingestão Bronze concluída para todos os arquivos.


In [0]:
# Validação pós-carga: lista as tabelas persistidas no schema bronze.
spark.sql("SHOW TABLES IN ecommerce_br.bronze").show()

+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
|  bronze|category_translation|      false|
|  bronze|           customers|      false|
|  bronze|         geolocation|      false|
|  bronze|         order_items|      false|
|  bronze|      order_payments|      false|
|  bronze|       order_reviews|      false|
|  bronze|              orders|      false|
|  bronze|            products|      false|
|  bronze|             sellers|      false|
+--------+--------------------+-----------+



In [0]:
%sql SHOW TABLES IN ecommerce_br.bronze;

database,tableName,isTemporary
bronze,category_translation,false
bronze,customers,false
bronze,geolocation,false
bronze,order_items,false
bronze,order_payments,false
bronze,order_reviews,false
bronze,orders,false
bronze,products,false
bronze,sellers,false
